In [ ]:
# ! pip install matplotlib

In [ ]:
import os

from rsciio import edax, msa  # , tia

print(os.getcwd())
from pathlib import Path

import flatdict as fd

# plot the spectrum
import matplotlib.pyplot as plt
import numpy as np

prefix = f"{os.getcwd()}/2026-03-13-Au-colloidal-15nm-TEM-EDAX-EDXS/DB_Files/feldhoff/Cole/Mapping/Lsm"
file_paths = [
    # f"{prefix}/Au-colloidal-15nm/Au-colloidal-15nm/Area 1/fov20260313123122839.ipr",
    # f"{prefix}/Au-colloidal-15nm/Au-colloidal-15nm/Area 1/spectrum20260313123253519_0.spc"
    # f"{prefix}/Au-colloidal-15nm/Au-colloidal-15nm/Area 2/map20260313123910395_0.spd.lsd""
    # f"{prefix}/Au-colloidal-15nm/Au-colloidal-15nm/Area 3/Line20260313124513296_Det 1_0.lsd"
    # f"{prefix}/Au-colloidal-15nm/Au-colloidal-15nm/Area 3/Line20260313124513296_Det 1_0.Spc",
    f"{prefix}/Au-colloidal-15nm/Au-colloidal-15nm/Area 3/Line20260313124513296_Det 1_0_C  K.dat",
    # two blanks, likely bug in EDAX Team for single char elements adding a blank :)
    f"{prefix}/Au-colloidal-15nm/Au-colloidal-15nm/Area 3/Line20260313124513296_Det 1_0_Na K.dat",
    f"{prefix}/Au-colloidal-15nm/Au-colloidal-15nm/Area 3/Line20260313124513296_Det 1_0_O  K.dat",
]

# file_paths = [f"{prefix}/spectrum20260313122453695.spc"]
print(file_paths[0])

In [ ]:
for file_path in file_paths:
    if file_path.lower().endswith(".dat"):
        what = file_path.rsplit("_", 1)[1].rsplit(".", 1)[0]
        print(what)
        numpy_array = np.fromfile(file_path, dtype=np.uint32, count=-1)
        # compare number of spectra with file size, in the Area3 example
        # 23 spectra, file size 92 B, so 4 B per accumulated element count
        # IPR file format specification from EDAX is from 2006 and mentioning earlier versions
        # the dominant Windows OS version was XP and of those the majority of XP installations
        # were 32-bit (>95%), XP x64 existed but niche OS
        print(f"numpy_array {numpy_array}, {type(numpy_array)}, {numpy_array.dtype}")
        continue
    elif file_path.lower().endswith(".ipr"):
        s = edax.file_reader(file_path, lazy=False)
    elif file_path.lower().endswith(".lsd"):
        # if not os.path.islink(f"{os.getcwd()}{file_path}.spd"):
        #     os.symlink(
        #         Path(f"{os.getcwd()}{file_path}"),
        #         Path(f"{os.getcwd()}{file_path}.spd")
        #     )
        stem = file_path.rsplit(".", 1)[0]
        print(stem)
        # continue

        os.rename(f"{stem}.lsd", f"{stem}.lsd.spd")
        os.rename(f"{stem}.Spc", f"{stem}.spc")

        s = edax.file_reader(Path(f"{stem}.lsd.spd"), lazy=False)

        os.rename(f"{stem}.lsd.spd", f"{stem}.lsd")
        os.rename(f"{stem}.spc", f"{stem}.Spc")
        # if os.path.islink(
        #     Path(f"{os.getcwd()}{file_path}.spd")):
        #     os.remove(Path(f"{os.getcwd()}{file_path}.spd"))
    else:
        s = edax.file_reader(file_path, lazy=False)

    for entry in s:  # s is list
        if isinstance(entry, dict):
            print(f">>>>>> {entry.keys()}")
            for group in ["metadata", "original_metadata"]:
                print(f">>>>>> >>>>>> {group}")
                for key, val in fd.FlatDict(entry[group], delimiter="/").items():
                    print(f"{key}, {val}")
            if "data" in entry:
                print(
                    f"{type(entry['data'])}, {np.shape(entry['data'])}, {entry['data'].dtype}"
                )
                print(
                    f"np.min, np.max {np.min(entry['data'])}, {np.max(entry['data'])}"
                )
            if "axes" in entry:
                for idx, axis in enumerate(entry["axes"]):
                    print(f"{idx}, {axis}")
        # elif isinstance(entry, list):
        #     print(f">>>>>> list")
        #     for item in entry:
        #         print(f"{item}")
    del s

In [ ]:
plt.plot(np.linspace(0, 4095.0, num=4096, endpoint=True) / 4096 * 40.95, entry["data"])

In [ ]:
for file_path in file_paths:
    s = edax.file_reader(file_path, lazy=False)
    for entry in s:
        if isinstance(entry, dict):
            print(f">>>>>> {entry.keys()}")
            for group in ["metadata", "original_metadata"]:
                print(f">>>>>> >>>>>> {group}")
                for key, val in fd.FlatDict(entry[group], delimiter="/").items():
                    print(f"{key}, {val}")
            if "data" in entry:
                print(
                    f"{type(entry['data'])}, {np.shape(entry['data'])}, {entry['data'].dtype}"
                )
                print(
                    f"np.min, np.max {np.min(entry['data'])}, {np.max(entry['data'])}"
                )
            if "axes" in entry:
                for idx, axis in enumerate(entry["axes"]):
                    print(f"{idx}, {axis}")
    del s

In [ ]:
for suffix in ["C", "Na", "O"]:
    file_name = f"{os.getcwd()}/{prefix}/Au-colloidal-15nm/Au-colloidal-15nm/Area 3/Line20260313124513296_Det 1_0_{suffix}{' ' * (3 - len(suffix))}K.dat"
    file_size = os.path.getsize(file_name)
    # print(file_name)
    # print(file_size / 4)
    dat = np.fromfile(file_name, dtype=np.uint32, count=-1)
    print(dat)

In [ ]:
file_paths = [
    "../tia_emi_ser_playground/2026-03-13-Au-colloidal-15nm/export-EDXS/Au-colloidal-15nm_Area1_EDS-Spot1.msa"
]
for file_path in file_paths:
    s = msa.file_reader(file_path, lazy=False, encoding="latin-1")
    for entry in s:
        if isinstance(entry, dict):
            print(f">>>>>> {entry.keys()}")
            for group in ["metadata", "original_metadata"]:
                print(f">>>>>> >>>>>> {group}")
                for key, val in fd.FlatDict(entry[group], delimiter="/").items():
                    print(f"{key}, {val}")
            if "data" in entry:
                print(
                    f"{type(entry['data'])}, {np.shape(entry['data'])}, {entry['data'].dtype}"
                )
                print(
                    f"np.min, np.max {np.min(entry['data'])}, {np.max(entry['data'])}"
                )
            if "axes" in entry:
                for idx, axis in enumerate(entry["axes"]):
                    print(f"{idx}, {axis}")
    # del s

In [ ]:
plt.plot(np.linspace(0, 4095.0, num=4096, endpoint=True) * 0.010, s[0]["data"])